In [1]:
# Creacion archivos de prueba
with open("estadoinicial.txt", "w") as f:
    f.write("326\n504\n187")

with open("estadofinal.txt", "w") as f:
    f.write("847\n265\n310")

print("Archivos de prueba creados exitosamente.")

Archivos de prueba creados exitosamente.


In [2]:
import heapq

# Lee el archivo .txt y lo convierte en una tupla de tuplas 3x3
def cargar_estado(filepath):
    with open(filepath, 'r') as f:
        lineas = [linea.strip() for linea in f.readlines() if linea.strip()]
    matriz = []
    for linea in lineas:
        matriz.append(tuple(int(c) for c in linea))
    return tuple(matriz)

# Muestra la matriz 3x3
def imprimir_tablero(estado):
    for fila in estado:
        print(" ".join(str(elem) if elem != 0 else "_" for elem in fila))
    print()

# Obtiene el estado de la matriz
def obtener_posicion(estado, valor):
    for r in range(3):
        for c in range(3):
            if estado[r][c] == valor:
                return r, c
    return None

def heuristica_vecindad(estado, estado_final):
    distancia = 0
    movimientos = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    ]

    # Guardar vecinos esperados de cada ficha en el objetivo
    vecinos_objetivo = {}

    for r in range(3):
        for c in range(3):
            ficha = estado_final[r][c]
            if ficha != 0:
                vecinos = []
                for dr, dc in movimientos:
                    nr = r + dr
                    nc = c + dc
                    if 0 <= nr < 3 and 0 <= nc < 3:
                        vecino = estado_final[nr][nc]
                        if vecino != 0:
                            vecinos.append(vecino)
                vecinos_objetivo[ficha] = vecinos

    # Evaluar el estado actual
    for r in range(3):
        for c in range(3):
            ficha = estado[r][c]
            if ficha != 0:
                vecinos_esperados = vecinos_objetivo[ficha]
                for vecino in vecinos_esperados:
                    # posición actual del vecino
                    vr, vc = obtener_posicion(estado, vecino)

                    # posición donde debería estar alrededor de la ficha
                    fr, fc = obtener_posicion(estado_final, ficha)

                    # posición relativa del vecino
                    vfr, vfc = obtener_posicion(estado_final, vecino)

                    # distancia actual del vecino respecto a la posición esperada
                    distancia += abs(vr - vfr) + abs(vc - vfc)


    return distancia

def obtener_sucesores(estado):
    r_vacio, c_vacio = obtener_posicion(estado, 0)
    movimientos = [(-1, 0, 'Abajo'), (1, 0, 'Arriba'), (0, -1, 'Derecha'), (0, 1, 'Izquierda')]
    sucesores = []
    
    for dr, dc, direccion in movimientos:
        nr, nc = r_vacio + dr, c_vacio + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Crear nueva matriz clonada con el movimiento
            nueva_matriz = [list(fila) for fila in estado]
            # Intercambiar ficha con el espacio vacío
            nueva_matriz[r_vacio][c_vacio], nueva_matriz[nr][nc] = nueva_matriz[nr][nc], nueva_matriz[r_vacio][c_vacio]
            
            nuevo_estado = tuple(tuple(fila) for fila in nueva_matriz)
            ficha_movida = estado[nr][nc]
            sucesores.append((nuevo_estado, f"Mover ficha {ficha_movida} hacia {direccion}"))
            
    return sucesores

def resolver_8_puzzle(archivo_inicial, archivo_final):
    estado_inicial = cargar_estado(archivo_inicial)
    estado_final = cargar_estado(archivo_final)
    
    count = 0
    open_set = []
    h_inicial = heuristica_vecindad(estado_inicial, estado_final)
    
    # Insertar el nodo inicial
    heapq.heappush(open_set, (h_inicial, count, estado_inicial, 0, []))
    
    costos_g = {estado_inicial: 0}
    visitados = set()
    iteraciones = 0
    
    while open_set:
        iteraciones += 1
        _, _, actual, g, camino = heapq.heappop(open_set)
        
        if actual in visitados:
            continue
        visitados.add(actual)
        
        if actual == estado_final:
            return camino, iteraciones, actual
        
        for vecino, paso in obtener_sucesores(actual):
            nuevo_g = g + 1
            
            if vecino not in costos_g or nuevo_g < costos_g[vecino]:
                costos_g[vecino] = nuevo_g
                h = heuristica_vecindad(vecino, estado_final)
                f = nuevo_g + h
                count += 1
                heapq.heappush(open_set, (f, count, vecino, nuevo_g, camino + [(paso, vecino)]))
                
    return None, iteraciones, None

In [3]:
camino_solucion, total_iteraciones, final = resolver_8_puzzle("estadoinicial.txt", "estadofinal.txt")

print("==========================================")
print("     RESULTADOS DE LA BÚSQUEDA A*        ")
print("==========================================\n")
print(f"Número total de iteraciones realizadas: {total_iteraciones}\n")

if camino_solucion is not None:
    print(f"Solución encontrada en {len(camino_solucion)} pasos:\n")
    print("--- ESTADO INICIAL ---")
    imprimir_tablero(cargar_estado("estadoinicial.txt"))
    
    for i, (movimiento, estado) in enumerate(camino_solucion, 1):
        print(f"Paso {i}: {movimiento}")
        imprimir_tablero(estado)
else:
    print("No se encontro ninguna solucion.")

     RESULTADOS DE LA BÚSQUEDA A*        

Número total de iteraciones realizadas: 415

Solución encontrada en 26 pasos:

--- ESTADO INICIAL ---
3 2 6
5 _ 4
1 8 7

Paso 1: Mover ficha 5 hacia Derecha
3 2 6
_ 5 4
1 8 7

Paso 2: Mover ficha 3 hacia Abajo
_ 2 6
3 5 4
1 8 7

Paso 3: Mover ficha 2 hacia Izquierda
2 _ 6
3 5 4
1 8 7

Paso 4: Mover ficha 6 hacia Izquierda
2 6 _
3 5 4
1 8 7

Paso 5: Mover ficha 4 hacia Arriba
2 6 4
3 5 _
1 8 7

Paso 6: Mover ficha 7 hacia Arriba
2 6 4
3 5 7
1 8 _

Paso 7: Mover ficha 8 hacia Derecha
2 6 4
3 5 7
1 _ 8

Paso 8: Mover ficha 5 hacia Abajo
2 6 4
3 _ 7
1 5 8

Paso 9: Mover ficha 6 hacia Abajo
2 _ 4
3 6 7
1 5 8

Paso 10: Mover ficha 4 hacia Izquierda
2 4 _
3 6 7
1 5 8

Paso 11: Mover ficha 7 hacia Arriba
2 4 7
3 6 _
1 5 8

Paso 12: Mover ficha 8 hacia Arriba
2 4 7
3 6 8
1 5 _

Paso 13: Mover ficha 5 hacia Derecha
2 4 7
3 6 8
1 _ 5

Paso 14: Mover ficha 1 hacia Derecha
2 4 7
3 6 8
_ 1 5

Paso 15: Mover ficha 3 hacia Abajo
2 4 7
_ 6 8
3 1 5

Paso 16: Mo